# 骨龄检测系统完整研发方案（v2，复制自 `骨龄评估系统研发方案v2.md`）

---

## 一、项目背景

### 1. 业务目标

传入儿童左手手腕正面X光片，输出对应骨龄数值，辅助判断儿童生长空间，为医学干预提供依据。

### 2. 传统医生操作流程

1. 拿到X光片
2. 过滤出需要的13根指骨
3. 对每一节骨头根据经验划分发育等级
4. 按照RUSCHN国际记分方法统计得分
5. 最终计算得出骨龄

### 3. 项目需求特点

- 仅处理左手手腕正面X光片
- 无需区分左右手、正反面
- 输入为单张X光图像，输出为骨龄数值

### 4. 行业现状与参考

学术界主流采用"检测+分类"两阶段方案，MAE普遍在5~6个月水平，阅片时间从人工40秒降至3秒左右。端到端方案精度可更高（MAE约5个月），但可解释性弱，临床信任度不足。

---

## 二、业务流程

### 整体流程（检测 + 分类 + 计分）

```
输入X光图片 → 图像预处理 → 多类多目标检测（7类） → 位置过滤筛选13根目标骨头 → 
每根骨头对应分类模型判断发育等级 → 按RUSCHN记分法汇总得分 → 输出骨龄 + 可视化结果
```

### 各环节说明

**1. 图像预处理**

- 灰度化（加权平均法）
- 中值滤波/双边滤波去噪
- CLAHE自适应直方图均衡化（提升对比度，保留局部密度特征）
- 可选：位置校正（Graham算法+凸包缺陷检测），提升对手部摆放的鲁棒性

**2. 检测环节**

- 采用7类多目标检测方案
- 按骨头位置/类型划分为7个类别（桡骨、尺骨、远节指骨、中节指骨、近节指骨、掌骨、拇指）
- 检测后基于位置关系筛选出需要的13根骨头
- 基线模型：YOLOv8；优化方向：SAM空间注意力 + PIoU损失函数

**3. 分类环节**

- 13根骨头对应9个分类模型（特征相似、得分差异小的骨头合并）
- 基线模型：MobileNetV3 / ResNet18
- 优化方向：深度可分离卷积 + 混合空洞卷积 + ECAM通道注意力 + Focal loss

**4. 计分环节**

- 每根骨头根据发育等级映射对应得分
- 按RUSCHN国际记分方法汇总
- 男孩女孩计算公式不同（性别为已知输入信息）

**5. 结果输出**

- 骨龄数值
- 13根骨头的检测框可视化（标注名称和等级）
- 每根骨头的等级、得分明细表
- 置信度指标

---

## 三、技术方案选型

### 基准方案：7类检测 + 9个分类模型（两阶段）

**选择理由**：

- 与临床医生判断逻辑一致，可解释性强
- 每一步误差可定位，便于调优
- 业务变更灵活（换计分标准只需改最后一步）

### 已排除的方案

| 方案           | 排除原因                      |
| ------------ | ------------------------- |
| 直接检测13个骨头    | 不同指骨外观相似度高，模型学习困难         |
| 手动遮挡不需要的骨头   | 需额外检测模型，黑色无效像素影响后续模型效果    |
| 单类多目标检测+位置过滤 | 仅靠X/Y坐标排序容易排错，个体差异大，规则不可靠 |
| 纯端到端直接回归骨龄   | 黑箱预测，缺乏可解释性，不符合医疗业务逻辑     |

### 后续可探索的优化方案

| 方案              | 预期收益       | 适用阶段   |
| --------------- | ---------- | ------ |
| 分级检测（先大区再细分）    | 小骨头检测更准    | 第二阶段后  |
| 多任务端到端（检测+骨龄回归） | MAE更低、推理更快 | 第四阶段探索 |
| 自监督预训练+微调       | 数据不足时效果提升  | 数据量受限时 |

---

## 四、研发迭代路线

### 第一阶段：基线跑通（MVP验证）—— 1~2周

**目标**：用最少时间把完整流程跑通，验证技术方案可行性

**任务**：

1. 数据准备：VOC→YOLO格式转换、训练/验证集拆分、分类数据集整理
2. 预处理基线：CLAHE + 灰度化 + 基础去噪
3. 检测模型：YOLOv8训练7类检测，验证mAP
4. 分类模型：MobileNetV3/ResNet18训练9个分类模型
5. 流水线串联：检测→过滤→分类→计分→输出，计算整体MAE

**验收**：完整流程跑通，有baseline MAE数值

---

### 第二阶段：精度优化 —— 2~3周

**目标**：用最小改动把精度提上来，投入产出比最高

**任务**：

检测模型优化：

- 加入SAM空间注意力机制（预计mAP +3%）
- 换PIoU损失函数（小目标定位更准）
- 尝试YOLOv8m版本，评估精度/算力trade-off

分类模型优化：

- 加入Focal loss解决等级类别不平衡
- 加入ECAM/SE通道注意力
- 深度可分离卷积 + 混合空洞卷积（轻量化提精度）

数据增强升级：

- 增加旋转（±15°以内）
- 增加MixUp/CutMix混合增强
- 多尺度训练

过滤逻辑优化：

- 完善13根骨头位置过滤算法
- 漏检/误检容错处理
- 加入解剖学先验约束

**验收**：整体MAE比基线下降15%以上，检测mAP达0.95+

---

### 第三阶段：工程化与可解释性 —— 2周

**目标**：从"能跑"变成"能用"，达到可演示/可试用的程度

**任务**：

1. 模型封装与部署：统一类接口、模型预加载、批量推理支持
2. 结果可视化：检测框标注、等级得分明细表、骨龄对比展示
3. 可解释性增强：Grad-CAM热力图、关键贡献骨头标注、置信度指标
4. 错误分析系统：易判错骨头统计、分年龄段误差分布、误差样本自动归档

**验收**：有可演示的完整系统，结果可视化清晰

---

### 第四阶段：进阶探索（可选）

**方向A：多任务端到端优化** —— 检测+骨龄回归联合训练，保留检测框输出
**方向B：分级检测方案** —— 先检测大区域再精细检测，适合高分辨率图
**方向C：自监督预训练** —— 无标注数据预训练+少量标注微调
**方向D：系统产品化** —— Web/桌面端界面、医生人工修正、报告导出、病例管理

---

## 五、风险与应对

| 风险                   | 应对策略                         |
| -------------------- | ---------------------------- |
| 数据量不足（800张分摊到9个分类模型） | 数据增强 + 公开数据集（RSNA等）补充 + 迁移学习 |
| 等级标注一致性差（医生间差异）      | 多医生标注取共识 + 明确标注规范 + 标注质量抽检   |
| 小骨头（远节指骨等）漏检         | PIoU损失 + 多尺度训练 + 注意力机制       |
| 年龄段数据不平衡             | Focal loss + 重采样 + 分年龄段评估    |
| 手部摆放位置不一致            | 位置校正算法 + 旋转/平移数据增强           |

---

## 六、整体时间线

| 阶段            | 周期       | 关键产出                   |
| ------------- | -------- | ---------------------- |
| 第一阶段：基线跑通     | 1~2周     | 完整推理流水线 + baseline MAE |
| 第二阶段：精度优化     | 2~3周     | 优化后模型 + 误差分析报告         |
| 第三阶段：工程化与可解释性 | 2周       | 可演示系统 + 可视化结果          |
| **前三阶段合计**    | **5~7周** | **可用版本**               |
| 第四阶段：进阶探索     | 按需       | 各方向实验验证                |

# 骨龄评估系统 — 开发总结（截至 2026-08-06）

> 目标：读取左手腕 X 光片 → 输出骨龄（岁/月）。采用「检测 → 骨过滤 → 分类 → RUS 计分 → 骨龄」两阶段方案。

## 1. 数据准备
- `voc_to_yolo.py`：把 VOC 格式的 881 张手掌标注（7 类骨骼）转成 YOLO 格式，训练/验证 705/176（无类别泄漏）
- `prepare_classification.py`：构建 ImageFolder 分类结构，每个等级至少 4 张验证样本
- `data_audit.py`：数据质检（亮度/重复图/等级可分性），发现并删除 2 张标签冲突的桡骨图

## 2. 预处理（`preprocess.py`）
- 灰度 + 中值滤波(3) + **CLAHE 自适应直方图均衡化**（clip=2.0, grid=8×8）

## 3. 检测模型（YOLOv8n）
- `train_detection.py` 两阶段迁移学习（先冻结 10 层训 60 epoch，再全量微调 40 epoch）
- **mAP50 = 0.991**，P=0.996 R=0.994；权重 `runs/bone7_ft/weights/best.pt`

## 4. 分类模型（9 个 ResNet18）
- 每个关节一个分类器（DIP/DIPFirst/MCP/MCPFirst/MIP/PIP/PIPFirst/Radius/Ulna）
- 类权重 CE + 余弦退火 + 早停；**关键修复**：ImageFolder 按字符串排序导致等级错位，改用 `grade_list` 数值映射
- **类别不平衡**（各关节 4x~20x，见下方统计 cell）：用类权重 CE（`w_i = total/(n_class·count_i)`）+ 按等级分层划分（每等级 ≥4 张进 val、≥1 张进 train）处理；无 <2 张的极端等级

## 4b. CORN 序数回归（9 关节全部完成 ✅）
- 全部 9 关节改 CORN 序数回归（利用等级有序性，缓解不平衡、降低 MAE）
- 8 关节平均等级 MAE = **0.49**（含 Ulna 后 9 关节平均 0.62）；8/9 优于普通分类、1 持平（Ulna **2.48→1.67** 大幅改善）
- 明细见下方代码 cell（`summary_ordinal.csv`）

## 5. 骨过滤（`filter_bones.py`）
- 从 7 类检测结果选出 **RUS 13 块骨**（Radius/Ulna/MCP-1,3,5/PIP-1,3,5/MIP-3,5/DIP-1,3,5）
- 拇指侧判定 + 顺序保持匹配，验证 30/30 完整

## 6. 计分模块（`scoring.py`）
- 解析用户提供的官方表格：`骨发育等级对照表.csv`（RUS-CHN 计分）+ `骨龄评分参考表_TW3_RUS系列.csv`（骨龄换算）
- `bone_age_from_rus()` 返回（中值/下限/上限），越界截断

## 7. 端到端流水线（`pipeline.py`）
- 检测 → 13 骨过滤 → 分类 → RUS 计分 → 骨龄可视化
- **默认全部关节用 ordinal**（`--use-ce` 可切回普通分类做对比实验）

## 8. RSNA 验证 → 发现问题 → 数据驱动校准
- 用 RSNA 儿科骨龄挑战赛真实标签验证：881 张训练图全部来自该数据集
- **RUS 表硬查失败**：MAE = 53.5 月、相关 -0.32
  - 根因：arthrosis 部分骨等级标注与真实成熟度脱节（桡骨相关仅 +0.15，却是最大权重 210/1000，成为噪声）
- **数据驱动校准修复**（`calibrate.py`）：13 骨 RUS 得分特征 → GradientBoosting 回归骨龄
  - 2306 张标注图按年龄分层 85/15；模型 `models/bone_age_regressor.pkl`
- **校准对比**（同一测试集，seed=42）：
  - 普通分类特征：MAE = 13.22 月，相关 0.902
  - 全 ordinal 特征：MAE = 13.51 月，相关 0.894（差异 0.3 月在噪声内）
  - 选定 **全 ordinal + ordinal 校准模型**（特征与模型严格匹配，分类等级更准）
- 实测：14732 真实 5.8 岁 → 全 ordinal+校准 **6.05 岁（误差 3.0 月）**；RUS 表 10.5 岁（误差 56 月）

## 9. Git 进度
- `02cdac4` 全链路代码、`54502c9` TW3 表接入、`7aba69a` 数据驱动校准、`590ff1a` ordinal 完成+校准更新 — 均已推送 GitHub
- 注意：仓库根为 `D:\project\step1`，全局 `.gitignore` 有 `data/` 规则，标签/特征 CSV 需 `git add -f` 强制入库

## 下一步
- 等 RSNA 训练集图片下载完 → 扩充校准数据进一步降 MAE
- 全量 1425 验证集正式评估报告
- 论文配套：Grad-CAM 可解释性、端到端回归对比实验


In [1]:
# 类别分布统计：检查分类数据是否存在类别不平衡
# 处理措施：类权重 CE (w_i = total/(n_class*count_i)) + 按等级分层划分
from pathlib import Path

base = Path("Bone Age Assessment/datasets/classification_pre")
rows = []
for joint in sorted(p.name for p in base.iterdir() if p.is_dir()):
    c = {}
    for split in ["train", "val"]:
        d = base / joint / split
        for g in sorted(int(x.name) for x in d.iterdir() if x.is_dir()):
            n = len(list((d / str(g)).glob("*.png")))
            c[g] = c.get(g, 0) + n
    grades = sorted(c)
    cnts = [c[g] for g in grades]
    mx, mn, total = max(cnts), min(cnts), sum(cnts)
    low = [g for g in grades if c[g] < 2]
    rows.append((joint, total, len(grades), mn, mx, mx / mn, low))

print(f"{'关节':10s} {'总数':>5s} {'等级数':>4s} {'最少':>4s} {'最多':>4s} {'倍率':>7s}  <2张等级")
for joint, total, ng, mn, mx, ratio, low in rows:
    print(f"{joint:10s} {total:5d} {ng:4d} {mn:4d} {mx:4d} {ratio:6.1f}x  {low}")

关节            总数  等级数   最少   最多      倍率  <2张等级
DIP         1262   11   24  273   11.4x  []
DIPFirst     635   11    9  154   17.1x  []
MCP         1262   10   28  211    7.5x  []
MCPFirst     633   11   29  115    4.0x  []
MIP         1262   12   20  353   17.6x  []
PIP         1264   12   12  243   20.2x  []
PIPFirst     635   12   14  117    8.4x  []
Radius       646   14   15  154   10.3x  []
Ulna         632   12   20  189    9.4x  []


In [4]:
# 序数回归 vs 普通分类：9 关节等级 MAE 对比
import csv
from pathlib import Path

# 普通分类 baseline（训练日志记录）
ce_mae = {"DIP": 0.43, "DIPFirst": 0.49, "MCP": 0.74, "MCPFirst": 0.41,
          "MIP": 0.59, "PIP": 0.64, "PIPFirst": 0.54, "Radius": 0.71,
          "Ulna": 2.48}

ord_mae = {}
p = Path("Bone Age Assessment/models/classification/summary_ordinal.csv")
with open(p, encoding="utf-8-sig") as f:   # utf-8-sig 兼容 BOM
    for r in csv.DictReader(f):
        ord_mae[r["joint"]] = float(r["best_mae"])
ord_mae.setdefault("Ulna", 1.67)   # Ulna 单独训练过，summary 被覆盖，补录

print(f"{'关节':10s} {'普通分类':>8s} {'Ordinal':>8s} {'提升':>6s}")
total = 0
for j in ["DIP", "DIPFirst", "MCP", "MCPFirst", "MIP", "PIP", "PIPFirst", "Radius", "Ulna"]:
    ce, od = ce_mae.get(j, float("nan")), ord_mae.get(j, float("nan"))
    delta = (ce - od) / ce * 100 if ce == ce else float("nan")
    total += od
    print(f"{j:10s} {ce:8.2f} {od:8.2f} {delta:5.0f}%")
print(f"\n9 关节 ordinal 平均等级 MAE = {total / 9:.4f}")

关节             普通分类  Ordinal     提升
DIP            0.43     0.40     8%
DIPFirst       0.49     0.42    15%
MCP            0.74     0.75    -1%
MCPFirst       0.41     0.36    12%
MIP            0.59     0.45    24%
PIP            0.64     0.55    15%
PIPFirst       0.54     0.42    22%
Radius         0.71     0.61    14%
Ulna           2.48     1.67    33%

9 关节 ordinal 平均等级 MAE = 0.6241
